In [6]:
import json
import pickle
import re
import unicodedata
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import sklearn_crfsuite
from sklearn_crfsuite import metrics as crf_metrics
from sklearn.model_selection import train_test_split
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 13})

### Access the data

In [12]:
data_dir = Path('..\..\datasets\processed_data')

with open(data_dir / 'processed.pkl', 'rb') as f:
    processed = pickle.load(f)

with open(data_dir / 'vocabs.pkl', 'rb') as f:
    v = pickle.load(f)
    token2id   = v['token2id'];   id2token   = v['id2token']
    label2id   = v['label2id'];   id2label   = v['id2label']
    char2id    = v['char2id'];    id2char    = v['id2char']
    PAD_ID     = v['PAD_ID'];     UNK_ID     = v['UNK_ID']
    PAD_LABEL_ID      = v['PAD_LABEL_ID']
    Entity_labels     = v['Entity_labels']
    MAX_LEN_LSTM      = v['MAX_LEN_LSTM']
    MAX_LEN_BERT      = v['MAX_LEN_BERT']

### Feature Engineering

In [13]:
# Lightweight regex POS tagger (no external dependency)
# Covers the most discriminative POS patterns for resume NER.
# In production, replace with spaCy: nlp(text).pos_

_POS_RULES = [
    (re.compile(r'^\d{4}$'),                  'CD_YEAR'),   # graduation year
    (re.compile(r'^\d+$'),                    'CD'),        # other digits
    (re.compile(r'^[A-Z][a-z]+$'),            'NNP'),       # TitleCase → proper noun
    (re.compile(r'^[A-Z]{2,}$'),              'NNP_ABB'),   # ALLCAPS → abbreviation
    (re.compile(r'.+ing$',re.I),              'VBG'),       # gerund
    (re.compile(r'.+ed$',re.I),               'VBD'),       # past tense
    (re.compile(r'.+er$',re.I),               'NN_ER'),     # developer, manager
    (re.compile(r'.+ly$',re.I),               'RB'),        # adverb
    (re.compile(r'.+tion$|.+sion$',re.I),     'NN_TION'),   # graduation, solution
    (re.compile(r'.+ment$',re.I),             'NN_MENT'),   # management
    (re.compile(r'@'),                        'EMAIL'),
    (re.compile(r'[+#]|\d'),                  'TECH'),      # C++, HTML5
]

def regex_pos(token: str) -> str:
    for pattern, tag in _POS_RULES:
        if pattern.search(token):
            return tag
    return 'NN'   # default: common noun


# Dual-token feature 

def align_raw_norm(raw_bio, norm_bio):
    """Pair each normalized token back to its original raw form."""
    aligned = []
    raw_iter = iter(raw_bio)
    for norm_tok, norm_tag in norm_bio:
        for raw_tok, raw_tag in raw_iter:
            if raw_tag == norm_tag:
                aligned.append((raw_tok, norm_tok, norm_tag))
                break
    return aligned


def token_features(raw_tok: str, norm_tok: str,
                   idx: int, raw_seq: list, norm_seq: list) -> dict:
    """
    Build CRF feature dictionary for one token.

    Orthographic flags  -> raw_tok  (casing / punctuation intact)
    Prefix/suffix       -> norm_tok (consistent normalized form)
    POS tag             -> raw_tok  (regex-based)
    Context window      -> both raw neighbours
    """
    prev_raw = raw_seq[idx - 1] if idx > 0 else '<START>'
    next_raw = raw_seq[idx + 1] if idx < len(raw_seq) - 1 else '<END>'

    feat = {
        # Identity 
        'token'            : norm_tok,
        'pos'              : regex_pos(raw_tok),

        # Orthographic (raw) 
        'is_title_case'    : raw_tok.istitle(),
        'is_all_upper'     : raw_tok.isupper(),
        'is_all_lower'     : raw_tok.islower(),
        'is_digit'         : raw_tok.isdigit(),
        'has_digit'        : any(c.isdigit() for c in raw_tok),
        'is_alnum'         : raw_tok.isalnum(),
        'has_hyphen'       : '-' in raw_tok,
        'has_at'           : '@' in raw_tok,
        'has_dot'          : '.' in raw_tok,
        'has_plus'         : '+' in raw_tok,
        'has_slash'        : '/' in raw_tok,

        # Shape 
        'token_length'     : len(raw_tok),
        'is_short'         : len(raw_tok) <= 2,

        # Prefix / suffix (normalized) 
        'prefix2'          : norm_tok[:2],
        'prefix3'          : norm_tok[:3],
        'prefix4'          : norm_tok[:4],
        'suffix2'          : norm_tok[-2:],
        'suffix3'          : norm_tok[-3:],
        'suffix4'          : norm_tok[-4:],

        # Context bigrams (raw)
        'prev_token'       : prev_raw.lower(),
        'prev_is_title'    : prev_raw.istitle(),
        'prev_is_upper'    : prev_raw.isupper(),
        'prev_pos'         : regex_pos(prev_raw),
        'next_token'       : next_raw.lower(),
        'next_is_title'    : next_raw.istitle(),
        'next_is_upper'    : next_raw.isupper(),
        'next_pos'         : regex_pos(next_raw),

        # Position
        'is_sentence_start': idx == 0,
        'is_sentence_end'  : idx == len(raw_seq) - 1,
    }
    return feat


def sequence_features(raw_bio, norm_bio):
    aligned   = align_raw_norm(raw_bio, norm_bio)
    raw_toks  = [r for r, _, _ in aligned]
    norm_toks = [n for _, n, _ in aligned]
    return [
        token_features(raw_toks[i], norm_toks[i], i, raw_toks, norm_toks)
        for i in range(len(aligned))
    ]

# Demo
demo_feats = sequence_features(
    processed[0]['word_bio_raw'],
    processed[0]['word_bio_norm']
)
show_keys = ['token','pos','is_title_case','is_all_upper','is_digit','has_at','prefix3','suffix3','prev_token','next_token']
demo_tags = [tag for _,tag in processed[0]['word_bio_norm']]

print(f"{'BIO Tag':<28}" + ''.join(f"{k:<18}" for k in show_keys))
print("=" * (28 + 18 * len(show_keys)))
for tag, feat in zip(demo_tags[:10], demo_feats[:10]):
    print(f"{tag:<28}" + ''.join(f"{str(feat[k]):<18}" for k in show_keys))

BIO Tag                     token             pos               is_title_case     is_all_upper      is_digit          has_at            prefix3           suffix3           prev_token        next_token        
B-Name                      abhishek          NNP               True              False             False             False             abh               hek               <start>           jha               
I-Name                      jha               NNP               True              False             False             False             jha               jha               abhishek          application       
B-Designation               application       NNP               True              False             False             False             app               ion               jha               development       
I-Designation               development       NNP               True              False             False             False             dev               ent       